# Assignment 3 — Regularization and Stability

1. Core functions: OLS, ridge, stability estimation
2. Self-tests
3. Datasets (Diabetes and synthetic)
4. Experiment 1: train/test error vs λ
5. Experiment 2: stability vs λ, and stability vs generalisation gap

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

## 1. Core functions

In [2]:
# Preprocessing

def standardize(X_train, X_test):

    mu = X_train.mean(axis=0)
    sd = X_train.std(axis=0)
    sd[sd == 0] = 1.0
    return (X_train - mu) / sd, (X_test - mu) / sd

In [3]:
# Ordinary least squares (unregularised baseline)

def ols_fit(X, y, cond_tol=1e10):

    mu_x, mu_y = X.mean(axis=0), y.mean()
    Xc, yc = X - mu_x, y - mu_y
    G = Xc.T @ Xc
    if np.linalg.matrix_rank(G) == G.shape[0] and np.linalg.cond(G) < cond_tol:
        w = np.linalg.solve(G, Xc.T @ yc)
    else:
        U, sv, Vt = np.linalg.svd(Xc, full_matrices=False)
        keep = sv > sv.max() * max(Xc.shape) * np.finfo(float).eps
        w = Vt[keep].T @ ((U[:, keep].T @ yc) / sv[keep])
    b = mu_y - mu_x @ w
    return w, b


In [4]:
# Ridge regression (closed form)

def ridge_fit(X, y, lam, m_ref=None):

    m, d = X.shape
    if lam == 0:
        return ols_fit(X, y)                            # unregularised baseline
    m_ref = m if m_ref is None else m_ref
    mu_x, mu_y = X.mean(axis=0), y.mean()
    Xc, yc = X - mu_x, y - mu_y
    alpha = m_ref * lam / 2.0
    w = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(d), Xc.T @ yc)
    b = mu_y - mu_x @ w
    return w, b


In [5]:
def predict(model, X):
    w, b = model
    return X @ w + b


def mse(y, y_hat):
    return float(np.mean((y - y_hat) ** 2))

In [6]:
# Stability estimation

def _summarise(pred_change, loss_gap, slope_change):
    return {
        "pred_mean": float(pred_change.mean()),   # Metric A, total (weights + intercept)
        "pred_max": float(pred_change.max()),
        "slope_mean": float(slope_change.mean()), # Metric A, penalised weights only
        "slope_max": float(slope_change.max()),
        "slope_change": slope_change,
        "loss_mean": float(loss_gap.mean()),      # Metric B (average)
        "loss_max": float(loss_gap.max()),        # Metric B (worst case)
        "pred_change": pred_change,               # per-point values
        "loss_gap": loss_gap,
    }


In [7]:
def stability_removal(X_tr, y_tr, X_te, lam):
    """
    Leave-one-out stability.
    For each training point i: retrain without it (penalty alpha kept fixed) and record
      pred_change[i] = mean_j | f_S(x_j) - f_{-i}(x_j) |  over the test set (total)
      slope_change[i]= mean_j | (x_j - xbar)^T (w_S - w_{-i}) |  (penalised weights only)
      loss_gap[i]    = l(f_{-i}, z_i) - l(f_S, z_i)      (squared loss on the removed point)
    NOTE: the intercept is unpenalised, so pred_change has a floor of about
    mean|y_i - ybar|/(m-1) even as lam -> infinity; slope_change has no such floor.
    """
    m = len(y_tr)
    full = ridge_fit(X_tr, y_tr, lam)
    pred_full = predict(full, X_te)
    pred_change, loss_gap, slope_change = np.empty(m), np.empty(m), np.empty(m)
    Xc_te = X_te - X_tr.mean(axis=0)
    for i in range(m):
        keep = np.arange(m) != i
        mod = ridge_fit(X_tr[keep], y_tr[keep], lam, m_ref=m)
        pred_change[i] = np.mean(np.abs(predict(mod, X_te) - pred_full))
        slope_change[i] = np.mean(np.abs(Xc_te @ (full[0] - mod[0])))
        xi = X_tr[i:i + 1]
        loss_gap[i] = ((y_tr[i] - predict(mod, xi)[0]) ** 2
                       - (y_tr[i] - predict(full, xi)[0]) ** 2)
    return _summarise(pred_change, loss_gap, slope_change)


def stability_replacement(X_tr, y_tr, X_te, lam, sampler, rng):
    """
    Replacement stability (synthetic data only, where D can be sampled).
    For each i: replace z_i by a fresh draw z'_i ~ D, retrain on S^(i), record
      pred_change[i] = mean_j | f_S(x_j) - f_{S^(i)}(x_j) |
      loss_gap[i]    = l(f_{S^(i)}, z_i) - l(f_S, z_i)   (z_i is the replaced point)
    `sampler(n, rng)` must return (X_new, y_new) with n fresh points from D.
    """
    m = len(y_tr)
    full = ridge_fit(X_tr, y_tr, lam)
    pred_full = predict(full, X_te)
    X_new, y_new = sampler(m, rng)
    pred_change, loss_gap, slope_change = np.empty(m), np.empty(m), np.empty(m)
    Xc_te = X_te - X_tr.mean(axis=0)
    for i in range(m):
        Xs, ys = X_tr.copy(), y_tr.copy()
        Xs[i], ys[i] = X_new[i], y_new[i]
        mod = ridge_fit(Xs, ys, lam, m_ref=m)
        pred_change[i] = np.mean(np.abs(predict(mod, X_te) - pred_full))
        slope_change[i] = np.mean(np.abs(Xc_te @ (full[0] - mod[0])))
        xi = X_tr[i:i + 1]
        loss_gap[i] = ((y_tr[i] - predict(mod, xi)[0]) ** 2
                       - (y_tr[i] - predict(full, xi)[0]) ** 2)
    return _summarise(pred_change, loss_gap, slope_change)


In [8]:
# Closed-form leave-one-out (used as a correctness check)

def loo_closed_form(X, y, lam):
    """
    Exact leave-one-out residuals for ridge with an unpenalised intercept and
    fixed penalty alpha = m*lam/2, via the hat matrix of the augmented design
    Z = [1, X]:   e_i^loo = e_i / (1 - H_ii).
    Returns (train_residuals, loo_residuals).
    """
    m, d = X.shape
    Z = np.hstack([np.ones((m, 1)), X])
    P = np.zeros((d + 1, d + 1))
    P[1:, 1:] = (m * lam / 2.0) * np.eye(d)
    A = Z.T @ Z + P
    Ainv = np.linalg.solve(A, np.eye(d + 1)) if lam > 0 else np.linalg.pinv(A)
    H = Z @ Ainv @ Z.T
    e = y - H @ y
    return e, e / (1.0 - np.diag(H))



## 2. Self-tests
Checks ridge/OLS correctness and the stability loop against a closed-form leave-one-out formula.

In [9]:
# Self-tests

def _toy_data(rng, n, d=10, rho=0.9, sigma=1.0, w_true=None):
    idx = np.arange(d)
    Sigma = rho ** np.abs(idx[:, None] - idx[None, :])
    X = rng.multivariate_normal(np.zeros(d), Sigma, size=n)
    y = X @ w_true + sigma * rng.standard_normal(n)
    return X, y

In [10]:
def run_self_tests():
    rng = np.random.default_rng(0)
    d, m = 10, 60
    w_true = rng.standard_normal(d)
    X_raw, y = _toy_data(rng, m, d, w_true=w_true)
    X_te_raw, y_te = _toy_data(rng, 2000, d, w_true=w_true)
    mu_raw, sd_raw = X_raw.mean(0), X_raw.std(0)
    X, X_te = standardize(X_raw, X_te_raw)

    # Test 1: the solution satisfies the first-order optimality condition.
    lam = 0.5
    w, b = ridge_fit(X, y, lam)
    Xc, yc = X - X.mean(0), y - y.mean()
    grad = (2.0 / m) * Xc.T @ (Xc @ w - yc) + lam * w
    assert np.allclose(grad, 0, atol=1e-10), "ridge_fit gradient is not zero"
    print("Test 1 passed: ridge_fit satisfies the optimality condition")

    # Test 2: retraining loop matches the closed-form leave-one-out loss gap.
    for lam in [0.0, 0.01, 0.5, 10.0]:
        res = stability_removal(X, y, X_te, lam)
        e, e_loo = loo_closed_form(X, y, lam)
        gap_closed = e_loo ** 2 - e ** 2
        assert np.allclose(res["loss_gap"], gap_closed, atol=1e-6), f"mismatch at lam={lam}"
    print("Test 2 passed: retraining loop == closed-form LOO (lam = 0, 0.01, 0.5, 10)")

    # Test 3: mean loss gap equals (LOO error - training error).
    res = stability_removal(X, y, X_te, 0.5)
    e, e_loo = loo_closed_form(X, y, 0.5)
    assert np.isclose(res["loss_mean"], np.mean(e_loo ** 2) - np.mean(e ** 2))
    print("Test 3 passed: mean loss gap == LOO error - training error")

    # Test 4: stability improves as lambda grows (penalised weights), and the
    # total metric hits the predicted intercept floor at huge lambda.
    lams = [0.0, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]
    out = [stability_removal(X, y, X_te, l) for l in lams]
    print("\nlambda      beta_total   beta_slope")
    for l, o in zip(lams, out):
        print(f"{l:<10g}  {o['pred_mean']:.5f}      {o['slope_mean']:.5f}")
    sl = [o["slope_mean"] for o in out]
    assert all(sl[i] >= sl[i + 1] - 1e-12 for i in range(len(sl) - 1)), "slope metric not decreasing"
    print("Test 4a passed: slope-only metric is non-increasing in lambda")
    big = stability_removal(X, y, X_te, 1e8)
    floor = np.mean(np.abs(y - y.mean())) / (m - 1)
    assert np.isclose(big["pred_mean"], floor, rtol=1e-3) and big["slope_mean"] < 1e-6
    print(f"Test 4b passed: total metric floors at mean|y-ybar|/(m-1) = {floor:.5f}; slope metric -> 0")

    # Test 5: replacement protocol runs and gives a comparable magnitude.
    def sampler(n, r):                      # fresh draws from D, scaled with TRAINING stats
        Xn, yn = _toy_data(r, n, d, w_true=w_true)
        return (Xn - mu_raw) / sd_raw, yn
    rep = stability_replacement(X, y, X_te, 0.5, sampler, np.random.default_rng(1))
    rem = stability_removal(X, y, X_te, 0.5)
    print(f"\nlam=0.5  removal beta_pred = {rem['pred_mean']:.5f} | replacement beta_pred = {rep['pred_mean']:.5f}")

    # Test 6: our ols_fit agrees with numpy's lstsq, both when X^T X is invertible
    # and when it is singular (d > m, minimum-norm solution that interpolates).
    for (n_, d_) in [(60, 10), (20, 30)]:
        Xr = rng.standard_normal((n_, d_)); yr = rng.standard_normal(n_)
        w_o, b_o = ols_fit(Xr, yr)
        Xc_, yc_ = Xr - Xr.mean(0), yr - yr.mean()
        w_ref = np.linalg.lstsq(Xc_, yc_, rcond=None)[0]
        assert np.allclose(w_o, w_ref, atol=1e-8), f"ols_fit mismatch for n={n_}, d={d_}"
        if d_ > n_:
            assert mse(yr, predict((w_o, b_o), Xr)) < 1e-20, "d > m should interpolate"
    print("Test 6 passed: ols_fit == lstsq (full rank) and min-norm interpolator (d > m)")

    # Test 7: ridge converges to OLS as lam -> 0+ (also in the singular case).
    for (n_, d_) in [(60, 10), (20, 30)]:
        Xr = rng.standard_normal((n_, d_)); yr = rng.standard_normal(n_)
        w_o, _ = ols_fit(Xr, yr)
        w_r, _ = ridge_fit(Xr, yr, 1e-9)
        assert np.allclose(w_o, w_r, atol=1e-4), f"ridge(lam->0) != OLS for n={n_}, d={d_}"
    print("Test 7 passed: ridge(lam -> 0+) converges to OLS")
    print("All checks completed.")

In [11]:
run_self_tests()

Test 1 passed: ridge_fit satisfies the optimality condition
Test 2 passed: retraining loop == closed-form LOO (lam = 0, 0.01, 0.5, 10)
Test 3 passed: mean loss gap == LOO error - training error

lambda      beta_total   beta_slope
0           0.03632      0.03500
0.001       0.03607      0.03474
0.01        0.03407      0.03267
0.1         0.02598      0.02391
1           0.02380      0.01803
10          0.02431      0.01027
100         0.02585      0.00239
Test 4a passed: slope-only metric is non-increasing in lambda
Test 4b passed: total metric floors at mean|y-ybar|/(m-1) = 0.02666; slope metric -> 0

lam=0.5  removal beta_pred = 0.02368 | replacement beta_pred = 0.03615
Test 6 passed: ols_fit == lstsq (full rank) and min-norm interpolator (d > m)
Test 7 passed: ridge(lam -> 0+) converges to OLS
All checks completed.


## 3. Datasets

In [12]:
# Real-world dataset: Diabetes

def split_diabetes(seed, m=100, n_test=None):

    X, y = load_diabetes(scaled=False, return_X_y=True)   # raw, unscaled features
    perm = np.random.default_rng(seed).permutation(len(y))
    if n_test is None:
        tr, te = perm[:m], perm[m:]
    else:
        te, tr = perm[:n_test], perm[n_test:n_test + m]
    X_tr, X_te = standardize(X[tr], X[te])                # training statistics only
    y_scale = y[tr].std()
    return {"X_tr": X_tr, "y_tr": y[tr] / y_scale,
            "X_te": X_te, "y_te": y[te] / y_scale,
            "y_scale": y_scale, "bayes": None, "sampler": None}


In [13]:
# Synthetic dataset

def synthetic_params(d=10, rho=0.96, r2=0.5, w_seed=12345):

    idx = np.arange(d)
    Sigma = rho ** np.abs(idx[:, None] - idx[None, :])
    w_true = np.random.default_rng(w_seed).standard_normal(d)
    sigma = np.sqrt((w_true @ Sigma @ w_true) * (1.0 / r2 - 1.0))
    return {"Sigma": Sigma, "w_true": w_true, "sigma": sigma, "d": d}


def sample_synthetic(params, n, rng):
    X = rng.multivariate_normal(np.zeros(params["d"]), params["Sigma"], size=n)
    y = X @ params["w_true"] + params["sigma"] * rng.standard_normal(n)
    return X, y


def split_synthetic(seed, m=100, n_test=5000, **kw):

    p = synthetic_params(**kw)
    rng = np.random.default_rng(seed)
    X_raw, y_raw = sample_synthetic(p, m, rng)
    Xte_raw, yte_raw = sample_synthetic(p, n_test, rng)
    mu, sd = X_raw.mean(axis=0), X_raw.std(axis=0)
    y_scale = y_raw.std()
    X_tr, X_te = standardize(X_raw, Xte_raw)

    def sampler(n, r):    # fresh draws from D, preprocessed with the TRAINING statistics
        Xn, yn = sample_synthetic(p, n, r)
        return (Xn - mu) / sd, yn / y_scale

    return {"X_tr": X_tr, "y_tr": y_raw / y_scale,
            "X_te": X_te, "y_te": yte_raw / y_scale,
            "y_scale": y_scale,
            "bayes": p["sigma"] ** 2 / y_scale ** 2,   # irreducible error, same units
            "sampler": sampler}


SPLITS = {"Diabetes (real)": split_diabetes, "Synthetic": split_synthetic}


In [14]:
# Dataset summary: the numbers quoted in Section 3 of the report
X_all, y_all = load_diabetes(scaled=False, return_X_y=True)
print("Diabetes: n = %d patients, d = %d features" % X_all.shape)
print("condition number of the feature correlation matrix, Diabetes : %.0f"
      % np.linalg.cond(np.corrcoef(X_all, rowvar=False)))
p = synthetic_params()
print("condition number of Sigma, synthetic                         : %.0f" % np.linalg.cond(p["Sigma"]))
signal = p["w_true"] @ p["Sigma"] @ p["w_true"]
print("population R^2 of the synthetic data                         : %.2f" % (signal / (signal + p["sigma"] ** 2)))

dd, ss = split_diabetes(0, 100), split_synthetic(0, 100)
print("Diabetes  split (m=100): train %s, test %s" % (dd["X_tr"].shape, dd["X_te"].shape))
print("Synthetic split (m=100): train %s, test %s" % (ss["X_tr"].shape, ss["X_te"].shape))
print("Bayes risk, synthetic (fraction of training label variance, mean of 50 seeds): %.4f"
      % np.mean([split_synthetic(s, 100)["bayes"] for s in range(50)]))

print("\nOLS test R^2 on Diabetes, R^2 = 1 - MSE / Var(y_test), 50 random splits:")
for m_ in [60, 100, 200, 353]:
    r2 = []
    for s in range(50):
        d_ = split_diabetes(s, m_)
        w_ = ols_fit(d_["X_tr"], d_["y_tr"])
        r2.append(1 - mse(d_["y_te"], predict(w_, d_["X_te"])) / np.var(d_["y_te"]))
    print("  m = %3d: mean %.3f (sd %.3f)" % (m_, np.mean(r2), np.std(r2)))


Diabetes: n = 442 patients, d = 10 features
condition number of the feature correlation matrix, Diabetes : 470
condition number of Sigma, synthetic                         : 420
population R^2 of the synthetic data                         : 0.50
Diabetes  split (m=100): train (100, 10), test (342, 10)
Synthetic split (m=100): train (100, 10), test (5000, 10)


Bayes risk, synthetic (fraction of training label variance, mean of 50 seeds): 0.5105

OLS test R^2 on Diabetes, R^2 = 1 - MSE / Var(y_test), 50 random splits:
  m =  60: mean 0.392 (sd 0.054)


  m = 100: mean 0.445 (sd 0.036)


  m = 200: mean 0.467 (sd 0.040)
  m = 353: mean 0.470 (sd 0.068)


## 4. Experiment 1 — training and test error vs λ

In [15]:
M_ERR = 100                          # training set size
N_SEEDS_ERR = 50
LAMBDAS_ERR = np.logspace(-4, 3, 30)  # lambda = 0 (OLS) is shown as a dashed reference


def run_errors(split_fn):
    tr = np.empty((N_SEEDS_ERR, len(LAMBDAS_ERR))); te = np.empty_like(tr)
    ols_tr, ols_te, bayes = np.empty(N_SEEDS_ERR), np.empty(N_SEEDS_ERR), []
    for s in range(N_SEEDS_ERR):
        d = split_fn(seed=s, m=M_ERR)
        w = ols_fit(d["X_tr"], d["y_tr"])
        ols_tr[s] = mse(d["y_tr"], predict(w, d["X_tr"]))
        ols_te[s] = mse(d["y_te"], predict(w, d["X_te"]))
        for k, lam in enumerate(LAMBDAS_ERR):
            w = ridge_fit(d["X_tr"], d["y_tr"], lam)
            tr[s, k] = mse(d["y_tr"], predict(w, d["X_tr"]))
            te[s, k] = mse(d["y_te"], predict(w, d["X_te"]))
        if d["bayes"] is not None:
            bayes.append(d["bayes"])
    return tr, te, ols_tr, ols_te, (np.mean(bayes) if bayes else None)


def sem(a, axis=0):
    return a.std(axis=axis, ddof=1) / np.sqrt(a.shape[axis])


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)
for ax, (name, fn) in zip(axes, SPLITS.items()):
    tr, te, ols_tr, ols_te, bayes = run_errors(fn)
    mte, mtr = te.mean(0), tr.mean(0)
    ax.plot(LAMBDAS_ERR, mte, color="C3", label="ridge: test")
    ax.fill_between(LAMBDAS_ERR, mte - sem(te), mte + sem(te), color="C3", alpha=0.2)
    ax.plot(LAMBDAS_ERR, mtr, color="C0", label="ridge: train")
    ax.fill_between(LAMBDAS_ERR, mtr - sem(tr), mtr + sem(tr), color="C0", alpha=0.2)
    ax.axhline(ols_te.mean(), color="C3", ls="--", label="OLS: test")
    ax.axhline(ols_tr.mean(), color="C0", ls="--", label="OLS: train")
    if bayes is not None:
        ax.axhline(bayes, color="k", ls=":", label="Bayes risk")
    ax.set_xscale("log"); ax.set_xlabel(r"regularisation strength $\lambda$")
    ax.set_ylabel("MSE (fraction of training label variance)")
    ax.set_title(f"{name}, m={M_ERR}, {N_SEEDS_ERR} seeds"); ax.legend(fontsize=8)

    k = int(np.argmin(mte))
    gain = 100 * (ols_te.mean() - mte[k]) / ols_te.mean()
    print(f"\n== {name} ==")
    print(f"OLS    : train {ols_tr.mean():.4f}   test {ols_te.mean():.4f} (+/- {sem(ols_te[:, None], 0)[0]:.4f})")
    print(f"ridge* : train {mtr[k]:.4f}   test {mte[k]:.4f}   at lambda = {LAMBDAS_ERR[k]:.3g}   "
          f"(test error {gain:.1f}% lower than OLS)")
    if bayes is not None:
        print(f"Bayes risk (irreducible): {bayes:.4f}")
    print(f"lambda -> max ({LAMBDAS_ERR[-1]:.0f}): train {mtr[-1]:.4f}  test {mte[-1]:.4f}")
plt.tight_layout()
plt.savefig("fig_error_vs_lambda.png", dpi=160)
plt.show()
print("\nsaved fig_error_vs_lambda.png")



== Diabetes (real) ==
OLS    : train 0.4509   test 0.5650 (+/- 0.0100)
ridge* : train 0.4664   test 0.5452   at lambda = 0.418   (test error 3.5% lower than OLS)
lambda -> max (1000): train 0.9940  test 1.0268



== Synthetic ==
OLS    : train 0.4421   test 0.5723 (+/- 0.0132)
ridge* : train 0.4818   test 0.5319   at lambda = 0.728   (test error 7.1% lower than OLS)
Bayes risk (irreducible): 0.5105
lambda -> max (1000): train 0.9828  test 1.0085



saved fig_error_vs_lambda.png


## 5. Experiment 2 — stability vs λ and the stability–generalisation link

In [17]:
M_STAB = 100
N_SEEDS_STAB = 50
LAMBDAS_STAB = np.concatenate([[0.0], np.logspace(-4, 3, 30)])   # index 0 is OLS
KEYS = ["slope", "total", "loss", "slope_max", "gap", "train", "test"]


def run_stability(split_fn):
    R = {k: np.empty((N_SEEDS_STAB, len(LAMBDAS_STAB))) for k in KEYS}
    for s in range(N_SEEDS_STAB):
        d = split_fn(seed=s, m=M_STAB)
        for k, lam in enumerate(LAMBDAS_STAB):
            res = stability_removal(d["X_tr"], d["y_tr"], d["X_te"], lam)
            w = ridge_fit(d["X_tr"], d["y_tr"], lam)
            tr, te = mse(d["y_tr"], predict(w, d["X_tr"])), mse(d["y_te"], predict(w, d["X_te"]))
            R["slope"][s, k], R["total"][s, k] = res["slope_mean"], res["pred_mean"]
            R["loss"][s, k], R["slope_max"][s, k] = res["loss_mean"], res["slope_max"]
            R["train"][s, k], R["test"][s, k], R["gap"][s, k] = tr, te, te - tr
    return R


def sem(a):
    return a.std(axis=0, ddof=1) / np.sqrt(a.shape[0])


In [18]:
t0 = time.time()
results = {name: run_stability(fn) for name, fn in SPLITS.items()}
print(f"finished in {time.time() - t0:.0f}s")
np.savez("results_stability.npz",
         **{f"{n}|{k}": v for n, R in results.items() for k, v in R.items()}, lambdas=LAMBDAS_STAB)

lam = LAMBDAS_STAB[1:]                      # positive lambdas for log axes

finished in 28s


In [19]:
# Figure 1: stability vs lambda
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (name, R) in zip(axes, results.items()):
    for key, col, lab in [("slope", "C2", r"weights only (penalised part)"),
                          ("total", "C1", "total (incl. intercept)")]:
        m_, s_ = R[key].mean(0), sem(R[key])
        ax.plot(lam, m_[1:], color=col, label=lab)
        ax.fill_between(lam, (m_ - s_)[1:], (m_ + s_)[1:], color=col, alpha=0.2)
        ax.axhline(m_[0], color=col, ls="--", lw=1)          # OLS reference
    ref = R["slope"].mean(0)[-8]                                 # anchor 1/lambda line
    ax.plot(lam[-12:], ref * lam[-8] / lam[-12:], "k:", label=r"$\propto 1/\lambda$")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"regularisation strength $\lambda$")
    ax.set_ylabel(r"stability $\beta_{pred}$ (lower = more stable)")
    ax.set_title(f"{name}, m={M_STAB}, {N_SEEDS_STAB} seeds (dashed: OLS)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_stability_vs_lambda.png", dpi=160)
plt.show()


In [20]:
# Figure 2: stability vs generalisation gap (Theorem 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (name, R) in zip(axes, results.items()):
    g, b = R["gap"].mean(0), R["loss"].mean(0)
    sc = ax.scatter(g, b, c=np.log10(np.maximum(LAMBDAS_STAB, 1e-4)), cmap="viridis", s=28)
    ax.scatter(g[0], b[0], marker="s", s=70, facecolors="none", edgecolors="r", label="OLS ($\\lambda=0$)")
    lim = [0, max(g.max(), b.max()) * 1.05]
    ax.plot(lim, lim, "k:", lw=1, label="y = x")
    ax.set_xlabel("generalisation gap  (test MSE − train MSE)")
    ax.set_ylabel(r"$\beta_{loss}$ = LOO error − train error")
    ax.set_title(name); ax.legend(fontsize=8)
    plt.colorbar(sc, ax=ax, label=r"$\log_{10}\lambda$")
plt.tight_layout(); plt.savefig("fig_stability_vs_gap.png", dpi=160)
plt.show()


In [21]:
for name, R in results.items():
    print(f"\n== {name} ==")
    s_m = R["slope"].mean(0)
    print(f"weights-only beta_pred: OLS {s_m[0]:.4f} -> lam=1: {np.interp(1, lam, s_m[1:]):.4f} "
          f"-> lam=100: {np.interp(100, lam, s_m[1:]):.5f}")
    sel = lam >= 10
    slope_loglog = np.polyfit(np.log(lam[sel]), np.log(s_m[1:][sel]), 1)[0]
    print(f"log-log slope of beta_pred vs lambda for lambda >= 10: {slope_loglog:.2f} (theory: -1)")
    g, b = R["gap"].mean(0), R["loss"].mean(0)
    print(f"corr(beta_loss, gap) across lambda: {np.corrcoef(g, b)[0, 1]:.3f};  "
          f"at OLS: beta_loss={b[0]:.4f} vs gap={g[0]:.4f}")
    k = int(np.argmin(R['test'].mean(0)))
    print(f"lambda minimising mean test error: {LAMBDAS_STAB[k]:.3g}  (beta_pred there: {s_m[k]:.4f})")
    print(f"mean worst-case/average ratio (slope_max/slope_mean) at OLS: "
          f"{(R['slope_max'].mean(0)[0] / s_m[0]):.1f}")


== Diabetes (real) ==
weights-only beta_pred: OLS 0.0156 -> lam=1: 0.0090 -> lam=100: 0.00060
log-log slope of beta_pred vs lambda for lambda >= 10: -0.89 (theory: -1)
corr(beta_loss, gap) across lambda: 0.999;  at OLS: beta_loss=0.1186 vs gap=0.1141
lambda minimising mean test error: 0.418  (beta_pred there: 0.0109)
mean worst-case/average ratio (slope_max/slope_mean) at OLS: 4.5

== Synthetic ==
weights-only beta_pred: OLS 0.0158 -> lam=1: 0.0050 -> lam=100: 0.00092
log-log slope of beta_pred vs lambda for lambda >= 10: -0.73 (theory: -1)
corr(beta_loss, gap) across lambda: 0.998;  at OLS: beta_loss=0.1168 vs gap=0.1302
lambda minimising mean test error: 0.728  (beta_pred there: 0.0054)
mean worst-case/average ratio (slope_max/slope_mean) at OLS: 3.9


## 6. Choosing λ by cross-validation, and the sample-size extension
a.λ is chosen by 10-fold CV on the training set only; the test set is used once. An 'oracle' λ (picked on the test curve) is shown only to measure how optimistic that would be.

b.Extension: how stability depends on the training size m, at fixed λ.


In [22]:
def cv_select_lambda(X, y, lambdas, K, rng):

    m = len(y)
    folds = np.array_split(rng.permutation(m), K)
    cv_mse = np.zeros(len(lambdas))
    for k, lam in enumerate(lambdas):
        sq_err = 0.0
        for f in folds:
            keep = np.ones(m, dtype=bool)
            keep[f] = False
            model = ridge_fit(X[keep], y[keep], lam)
            sq_err += np.sum((y[f] - predict(model, X[f])) ** 2)
        cv_mse[k] = sq_err / m
    return lambdas[int(np.argmin(cv_mse))], cv_mse


In [23]:
# a. cross-validated lambda

CV_GRID = np.logspace(-4, 3, 30)
M_CV, N_SEEDS_CV, K_FOLDS = 100, 50, 10


def run_cv(split_fn):
    out = {k: np.empty(N_SEEDS_CV) for k in
           ["ols_te", "cv_te", "oracle_te", "lam_cv", "lam_oracle", "slope_ols", "slope_cv"]}
    for s in range(N_SEEDS_CV):
        d = split_fn(seed=s, m=M_CV)
        Xtr, ytr, Xte, yte = d["X_tr"], d["y_tr"], d["X_te"], d["y_te"]
        te_curve = np.array([mse(yte, predict(ridge_fit(Xtr, ytr, l), Xte)) for l in CV_GRID])
        lam_cv, _ = cv_select_lambda(Xtr, ytr, CV_GRID, K_FOLDS, np.random.default_rng(1000 + s))
        k_cv = int(np.argmin(np.abs(np.log(CV_GRID) - np.log(lam_cv))))
        out["ols_te"][s] = mse(yte, predict(ols_fit(Xtr, ytr), Xte))
        out["cv_te"][s] = te_curve[k_cv]
        out["oracle_te"][s] = te_curve.min()
        out["lam_cv"][s], out["lam_oracle"][s] = lam_cv, CV_GRID[int(np.argmin(te_curve))]
        out["slope_ols"][s] = stability_removal(Xtr, ytr, Xte, 0.0)["slope_mean"]
        out["slope_cv"][s] = stability_removal(Xtr, ytr, Xte, lam_cv)["slope_mean"]
    return out

# b. sample-size extension

M_GRID = [30, 50, 75, 100, 150, 200, 300]
LAM_M = [0.0, 0.01, 0.1, 1.0, 10.0]          # lambda = 0 is OLS
N_SEEDS_M, N_TEST_DIAB = 30, 100


def run_m(name):
    R = {k: np.empty((N_SEEDS_M, len(M_GRID), len(LAM_M))) for k in ["slope", "loss", "gap"]}
    for s in range(N_SEEDS_M):
        for i, m in enumerate(M_GRID):
            d = (split_diabetes(seed=s, m=m, n_test=N_TEST_DIAB) if name.startswith("Diab")
                 else split_synthetic(seed=s, m=m))
            for j, lam in enumerate(LAM_M):
                res = stability_removal(d["X_tr"], d["y_tr"], d["X_te"], lam)
                w = ridge_fit(d["X_tr"], d["y_tr"], lam)
                gap = (mse(d["y_te"], predict(w, d["X_te"])) - mse(d["y_tr"], predict(w, d["X_tr"])))
                R["slope"][s, i, j], R["loss"][s, i, j], R["gap"][s, i, j] = \
                    res["slope_mean"], res["loss_mean"], gap
    return R


def sem(a, axis=0):
    return a.std(axis=axis, ddof=1) / np.sqrt(a.shape[axis])


In [24]:
t0 = time.time()
names = ["Diabetes (real)", "Synthetic"]
fns = [split_diabetes, split_synthetic]

cv = {n: run_cv(f) for n, f in zip(names, fns)}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
for ax, n in zip(axes, names):
    o = cv[n]
    bins = np.logspace(-4, 3, 22)
    ax.hist(o["lam_cv"], bins=bins, alpha=0.7, label=r"$\lambda$ chosen by 10-fold CV")
    ax.hist(o["lam_oracle"], bins=bins, alpha=0.5, label=r"$\lambda$ minimising test error (oracle)")
    ax.set_xscale("log"); ax.set_xlabel(r"selected $\lambda$"); ax.set_ylabel("number of seeds")
    ax.set_title(f"{n}, m={M_CV}, {N_SEEDS_CV} seeds"); ax.legend(fontsize=8)
    print(f"\n== CV selection: {n} ==")
    print(f"median lambda: CV {np.median(o['lam_cv']):.3g} | oracle {np.median(o['lam_oracle']):.3g}")
    for lab, key in [("ridge, lambda by CV", "cv_te"), ("ridge, oracle lambda", "oracle_te")]:
        diff = o["ols_te"] - o[key]
        print(f"test MSE {lab:<22}: {o[key].mean():.4f}  | OLS {o['ols_te'].mean():.4f} | "
              f"paired gain {diff.mean():+.4f} +/- {sem(diff[:, None])[0]:.4f} (sem), "
              f"better in {(diff > 0).sum()}/{N_SEEDS_CV} seeds")
    print(f"stability (weights only): OLS {o['slope_ols'].mean():.4f} -> CV lambda {o['slope_cv'].mean():.4f}")
plt.tight_layout(); plt.savefig("fig_cv_lambda.png", dpi=160); plt.show()


res_m = {n: run_m(n) for n in names}
np.savez("results_sample_size.npz", m_grid=M_GRID, lam=LAM_M,
         **{f"{n}|{k}": v for n, R in res_m.items() for k, v in R.items()})
ms = np.array(M_GRID, dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, n in zip(axes, names):
    R = res_m[n]
    for j, lam in enumerate(LAM_M):
        mean, err = R["slope"][:, :, j].mean(0), sem(R["slope"][:, :, j])
        lab = "OLS ($\\lambda=0$)" if lam == 0 else f"$\\lambda={lam:g}$"
        ax.errorbar(ms, mean, yerr=err, marker="o", ms=4, capsize=2, label=lab)
    ref = res_m[n]["slope"][:, :, LAM_M.index(1.0)].mean(0)
    ax.plot(ms, ref[0] * ms[0] / ms, "k:", label=r"$\propto 1/m$")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("training set size $m$"); ax.set_ylabel(r"stability $\beta_{pred}$ (weights only)")
    ax.set_title(f"{n} (dotted: 1/m)"); ax.legend(fontsize=8)
    print(f"\n== sample-size sweep: {n} ==  (log-log slope of beta vs m, m in 50..300)")
    sel = (ms >= 50)
    for j, lam in enumerate(LAM_M):
        sl = np.polyfit(np.log(ms[sel]), np.log(R["slope"][:, :, j].mean(0)[sel]), 1)[0]
        g = R["gap"][:, :, j].mean(0)
        print(f"lambda={lam:<5g}: slope {sl:+.2f} | beta(m=30)={R['slope'][:, 0, j].mean():.4f} "
              f"beta(m=300)={R['slope'][:, -1, j].mean():.4f} | gap(m=30)={g[0]:.3f} gap(m=300)={g[-1]:.3f}")
plt.tight_layout(); plt.savefig("fig_stability_vs_m.png", dpi=160); plt.show()
print(f"\nfinished in {time.time() - t0:.0f}s")



== CV selection: Diabetes (real) ==
median lambda: CV 0.418 | oracle 0.418
test MSE ridge, lambda by CV   : 0.5537  | OLS 0.5650 | paired gain +0.0113 +/- 0.0039 (sem), better in 39/50 seeds
test MSE ridge, oracle lambda  : 0.5415  | OLS 0.5650 | paired gain +0.0235 +/- 0.0045 (sem), better in 44/50 seeds
stability (weights only): OLS 0.0156 -> CV lambda 0.0113

== CV selection: Synthetic ==
median lambda: CV 0.728 | oracle 0.728
test MSE ridge, lambda by CV   : 0.5352  | OLS 0.5723 | paired gain +0.0371 +/- 0.0031 (sem), better in 50/50 seeds
test MSE ridge, oracle lambda  : 0.5285  | OLS 0.5723 | paired gain +0.0438 +/- 0.0040 (sem), better in 50/50 seeds
stability (weights only): OLS 0.0158 -> CV lambda 0.0060



== sample-size sweep: Diabetes (real) ==  (log-log slope of beta vs m, m in 50..300)
lambda=0    : slope -1.15 | beta(m=30)=0.0831 beta(m=300)=0.0046 | gap(m=30)=0.543 gap(m=300)=0.047
lambda=0.01 : slope -1.12 | beta(m=30)=0.0707 beta(m=300)=0.0044 | gap(m=30)=0.468 gap(m=300)=0.045
lambda=0.1  : slope -1.09 | beta(m=30)=0.0582 beta(m=300)=0.0041 | gap(m=30)=0.379 gap(m=300)=0.041
lambda=1    : slope -1.04 | beta(m=30)=0.0335 beta(m=300)=0.0028 | gap(m=30)=0.249 gap(m=300)=0.032
lambda=10   : slope -1.02 | beta(m=30)=0.0121 beta(m=300)=0.0011 | gap(m=30)=0.168 gap(m=300)=0.023

== sample-size sweep: Synthetic ==  (log-log slope of beta vs m, m in 50..300)
lambda=0    : slope -1.13 | beta(m=30)=0.0791 beta(m=300)=0.0048 | gap(m=30)=0.538 gap(m=300)=0.034
lambda=0.01 : slope -1.09 | beta(m=30)=0.0614 beta(m=300)=0.0043 | gap(m=30)=0.421 gap(m=300)=0.031
lambda=0.1  : slope -1.04 | beta(m=30)=0.0350 beta(m=300)=0.0029 | gap(m=30)=0.254 gap(m=300)=0.020
lambda=1    : slope -1.02 | beta(m


finished in 36s
